# ProofLayer: Automated Cybersecurity Compliance Auditing for Maryland Government Policies

This notebook implements the full analysis pipeline used in the paper:
> *ProofLayer: Automated Cybersecurity Compliance Auditing for State Government Policies via Multi-LLM Orchestration*

It extracts text from Maryland government cybersecurity policy PDFs, applies keyword-based control mapping against FedRAMP, NIST 800-53, and CMMC 2.0 control families, computes compliance scores, performs gap analysis, and generates all figures used in the paper.

## 0. Setup and Dependencies

In [ ]:
# Install required packages if not present
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'pdfplumber', 'matplotlib', 'numpy', 'pandas'], check=True)

In [ ]:
import os
import json
import pdfplumber
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

matplotlib.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'figure.dpi': 150,
})

print('Libraries loaded successfully.')

## 1. Control Dataset (FedRAMP / NIST 800-53 / CMMC 2.0)

These 22 controls are drawn directly from `backend/controls.py` and span six NIST control families:
Access Control (AC), Configuration Management (CM), Audit and Accountability (AU),
Incident Response (IR), Risk Assessment (RA), System and Communications Protection (SC),
Personnel Security (PS), Contingency Planning (CP), Supply Chain (SR), and Media Protection (MP).

In [ ]:
CONTROL_DATASET = [
    # Access Control
    {"id": "AC-01", "control": "Access Control Policy & Procedures", "category": "Access Control",
     "keywords": ["access control policy", "access policy", "access procedures", "access management policy"],
     "frameworks": {"FedRAMP": "AC-1", "NIST_800_53": "AC-1", "CMMC": "AC.L1-3.1.1", "SOC2": "CC6.1"}, "risk": "HIGH"},
    {"id": "AC-02", "control": "Account Management", "category": "Access Control",
     "keywords": ["account management", "user accounts", "account lifecycle", "provisioning",
                  "deprovisioning", "user onboarding", "offboarding"],
     "frameworks": {"FedRAMP": "AC-2", "NIST_800_53": "AC-2", "CMMC": "AC.L1-3.1.1", "SOC2": "CC6.2"}, "risk": "HIGH"},
    {"id": "AC-03", "control": "Least Privilege / Role-Based Access", "category": "Access Control",
     "keywords": ["least privilege", "role-based access", "rbac", "minimum necessary",
                  "need to know", "permissions", "roles"],
     "frameworks": {"FedRAMP": "AC-6", "NIST_800_53": "AC-6", "CMMC": "AC.L1-3.1.2", "SOC2": "CC6.3"}, "risk": "HIGH"},
    {"id": "AC-04", "control": "Multi-Factor Authentication (MFA)", "category": "Access Control",
     "keywords": ["mfa", "multi-factor", "two-factor", "2fa", "authentication", "piv", "cac"],
     "frameworks": {"FedRAMP": "IA-2(1)", "NIST_800_53": "IA-2", "CMMC": "IA.L2-3.5.3", "SOC2": "CC6.1"}, "risk": "HIGH"},
    {"id": "AC-05", "control": "Remote Access Controls", "category": "Access Control",
     "keywords": ["remote access", "vpn", "remote desktop", "rdp", "ssh", "telework"],
     "frameworks": {"FedRAMP": "AC-17", "NIST_800_53": "AC-17", "CMMC": "AC.L2-3.1.12", "SOC2": "CC6.6"}, "risk": "MEDIUM"},
    # Configuration Management
    {"id": "CM-01", "control": "Configuration Management Policy", "category": "Configuration Management",
     "keywords": ["configuration management", "baseline configuration", "hardening",
                  "secure configuration", "system configuration"],
     "frameworks": {"FedRAMP": "CM-1", "NIST_800_53": "CM-1", "CMMC": "CM.L2-3.4.1", "SOC2": "CC7.1"}, "risk": "HIGH"},
    {"id": "CM-02", "control": "Change Management / Change Control", "category": "Configuration Management",
     "keywords": ["change management", "change control", "change request", "change approval", "cab"],
     "frameworks": {"FedRAMP": "CM-3", "NIST_800_53": "CM-3", "CMMC": "CM.L2-3.4.3", "SOC2": "CC8.1"}, "risk": "MEDIUM"},
    # Audit & Accountability
    {"id": "AU-01", "control": "Audit Logging & Monitoring", "category": "Audit & Accountability",
     "keywords": ["audit log", "logging", "log management", "siem", "monitoring",
                  "centralized logging", "event log"],
     "frameworks": {"FedRAMP": "AU-2", "NIST_800_53": "AU-2", "CMMC": "AU.L2-3.3.1", "SOC2": "CC7.2"}, "risk": "HIGH"},
    {"id": "AU-02", "control": "Audit Review & Alerting", "category": "Audit & Accountability",
     "keywords": ["audit review", "log review", "alert", "anomaly detection",
                  "threat detection", "monitoring alerts"],
     "frameworks": {"FedRAMP": "AU-6", "NIST_800_53": "AU-6", "CMMC": "AU.L2-3.3.5", "SOC2": "CC7.3"}, "risk": "HIGH"},
    # Incident Response
    {"id": "IR-01", "control": "Incident Response Plan", "category": "Incident Response",
     "keywords": ["incident response", "incident plan", "ir plan", "security incident",
                  "breach response", "incident handling"],
     "frameworks": {"FedRAMP": "IR-1", "NIST_800_53": "IR-1", "CMMC": "IR.L2-3.6.1", "SOC2": "CC7.4"}, "risk": "HIGH"},
    {"id": "IR-02", "control": "Incident Reporting", "category": "Incident Response",
     "keywords": ["incident reporting", "report incidents", "breach notification",
                  "us-cert", "cisa reporting"],
     "frameworks": {"FedRAMP": "IR-6", "NIST_800_53": "IR-6", "CMMC": "IR.L2-3.6.2", "SOC2": "CC7.4"}, "risk": "HIGH"},
    # Risk Assessment
    {"id": "RA-01", "control": "Risk Assessment", "category": "Risk Assessment",
     "keywords": ["risk assessment", "risk analysis", "risk evaluation", "threat modeling",
                  "vulnerability assessment", "risk register"],
     "frameworks": {"FedRAMP": "RA-3", "NIST_800_53": "RA-3", "CMMC": "RM.L2-3.11.1", "SOC2": "CC3.2"}, "risk": "HIGH"},
    {"id": "RA-02", "control": "Vulnerability Scanning", "category": "Risk Assessment",
     "keywords": ["vulnerability scan", "vulnerability management", "patch management",
                  "cve", "penetration test", "pen test"],
     "frameworks": {"FedRAMP": "RA-5", "NIST_800_53": "RA-5", "CMMC": "RM.L2-3.11.2", "SOC2": "CC7.1"}, "risk": "HIGH"},
    # System & Communications
    {"id": "SC-01", "control": "Data Encryption in Transit", "category": "System & Communications",
     "keywords": ["encryption in transit", "tls", "https", "ssl",
                  "transport encryption", "encrypted communications"],
     "frameworks": {"FedRAMP": "SC-8", "NIST_800_53": "SC-8", "CMMC": "SC.L2-3.13.8", "SOC2": "CC6.7"}, "risk": "HIGH"},
    {"id": "SC-02", "control": "Data Encryption at Rest", "category": "System & Communications",
     "keywords": ["encryption at rest", "disk encryption", "database encryption",
                  "aes", "data at rest", "encrypted storage"],
     "frameworks": {"FedRAMP": "SC-28", "NIST_800_53": "SC-28", "CMMC": "SC.L2-3.13.16", "SOC2": "CC6.7"}, "risk": "HIGH"},
    {"id": "SC-03", "control": "Network Segmentation", "category": "System & Communications",
     "keywords": ["network segmentation", "firewall", "dmz", "vlan",
                  "network isolation", "zero trust", "microsegmentation"],
     "frameworks": {"FedRAMP": "SC-7", "NIST_800_53": "SC-7", "CMMC": "SC.L1-3.13.1", "SOC2": "CC6.6"}, "risk": "MEDIUM"},
    # Personnel Security
    {"id": "PS-01", "control": "Security Awareness Training", "category": "Personnel Security",
     "keywords": ["security training", "security awareness", "phishing training",
                  "user training", "security education"],
     "frameworks": {"FedRAMP": "AT-2", "NIST_800_53": "AT-2", "CMMC": "AT.L2-3.2.1", "SOC2": "CC1.4"}, "risk": "MEDIUM"},
    {"id": "PS-02", "control": "Personnel Screening / Background Checks", "category": "Personnel Security",
     "keywords": ["background check", "personnel screening", "vetting",
                  "clearance", "pre-employment screening"],
     "frameworks": {"FedRAMP": "PS-3", "NIST_800_53": "PS-3", "CMMC": "PS.L2-3.9.1", "SOC2": "CC1.4"}, "risk": "MEDIUM"},
    # Contingency Planning
    {"id": "CP-01", "control": "Backup & Recovery", "category": "Contingency Planning",
     "keywords": ["backup", "recovery", "disaster recovery", "business continuity",
                  "rpo", "rto", "data backup"],
     "frameworks": {"FedRAMP": "CP-9", "NIST_800_53": "CP-9", "CMMC": "RE.L2-3.8.9", "SOC2": "A1.2"}, "risk": "HIGH"},
    {"id": "CP-02", "control": "Contingency / Disaster Recovery Plan", "category": "Contingency Planning",
     "keywords": ["contingency plan", "disaster recovery plan", "drp", "bcp",
                  "business continuity plan"],
     "frameworks": {"FedRAMP": "CP-2", "NIST_800_53": "CP-2", "CMMC": "RE.L2-3.8.9", "SOC2": "A1.3"}, "risk": "HIGH"},
    # Supply Chain & Media
    {"id": "SR-01", "control": "Third-Party / Vendor Risk Management", "category": "Supply Chain",
     "keywords": ["vendor risk", "third-party risk", "supply chain", "vendor assessment",
                  "contractor risk", "third party"],
     "frameworks": {"FedRAMP": "SA-9", "NIST_800_53": "SR-1", "CMMC": "CM.L2-3.4.6", "SOC2": "CC9.2"}, "risk": "MEDIUM"},
    {"id": "MP-01", "control": "Media Sanitization", "category": "Media Protection",
     "keywords": ["media sanitization", "data destruction", "secure disposal",
                  "disk wiping", "degaussing"],
     "frameworks": {"FedRAMP": "MP-6", "NIST_800_53": "MP-6", "CMMC": "MP.L1-3.8.3", "SOC2": "CC6.5"}, "risk": "MEDIUM"},
]

print(f'Control dataset loaded: {len(CONTROL_DATASET)} controls across '
      f'{len(set(c["category"] for c in CONTROL_DATASET))} categories')

## 2. Policy Document Loading

We load 10 Maryland state and local government cybersecurity policy documents.
Adjust `POLICY_DIR` to point to your local copy of the policy PDFs.

In [ ]:
# Adjust this path to where the policy PDFs are stored
POLICY_DIR = '../policies'

POLICY_FILES = {
    'MD_DOIT_CyberRiskMgmt_Policy.pdf':       'MD Risk Mgmt Policy',
    'MD_DOIT_SystemNetworkSecurity_Policy.pdf': 'MD System & Network Security Policy',
    'MD_DOIT_ContinuousMonitoring_Policy.pdf': 'MD Continuous Monitoring Policy',
    'MD_CybersecurityCouncil_Report_2025.pdf': 'MD Cybersecurity Council Report 2025',
    'MD_IT_SecurityManual.pdf':                'MD IT Security Manual',
    'MD_UMGC_LocalGov_Cybersecurity_2021.pdf': 'MD Local Gov Cybersecurity 2021',
    'MD_DHMH_IT_SecurityPolicy.pdf':           'MD Health Dept IT Security Policy',
    'MD_Judicial_InfoSecurity_Policy.pdf':     'MD Judicial Info Security Policy',
    'MD_MSDE_AUP_2024.pdf':                   'MD MSDE Acceptable Use Policy 2024',
    'MD_Procurement_Manual.pdf':               'MD Procurement Manual',
}

def extract_text(filepath, max_pages=15):
    """Extract text from a PDF file, up to max_pages pages."""
    with pdfplumber.open(filepath) as pdf:
        pages = pdf.pages[:max_pages]
        return '\n'.join(p.extract_text() or '' for p in pages)

docs = {}
for filename, label in POLICY_FILES.items():
    filepath = os.path.join(POLICY_DIR, filename)
    if os.path.exists(filepath):
        text = extract_text(filepath)
        docs[label] = text
        print(f'Loaded {label}: {len(text):,} chars')
    else:
        print(f'WARNING: {filename} not found at {filepath}')

print(f'\nTotal documents loaded: {len(docs)}')

## 3. Control Mapping: Keyword-Based Stage

ProofLayer Stage 1 applies fast keyword matching against the control dataset.
This stage is free (no LLM call) and runs in milliseconds per document.

In [ ]:
def keyword_map(text):
    """Map security controls to a document using keyword matching."""
    text_lower = text.lower()
    matched = []
    seen = set()
    for ctrl in CONTROL_DATASET:
        if ctrl['id'] in seen:
            continue
        if any(kw in text_lower for kw in ctrl['keywords']):
            matched.append(ctrl)
            seen.add(ctrl['id'])
    return matched

def compute_score(matched, total):
    """Compliance coverage score as percentage."""
    return min(100, int(len(matched) / max(total, 1) * 100))

def risk_level(score):
    if score >= 80: return 'LOW'
    if score >= 55: return 'MEDIUM'
    return 'HIGH'

results = {}
for doc_name, text in docs.items():
    matched = keyword_map(text)
    matched_ids = {c['id'] for c in matched}
    missing = [c for c in CONTROL_DATASET if c['id'] not in matched_ids]
    score = compute_score(matched, len(CONTROL_DATASET))
    results[doc_name] = {
        'score': score,
        'risk_level': risk_level(score),
        'matched': len(matched),
        'missing': len(missing),
        'matched_ids': sorted(matched_ids),
        'missing_ids': sorted(c['id'] for c in missing),
        'missing_high_risk': sorted(c['id'] for c in missing if c['risk'] == 'HIGH'),
    }
    print(f"{doc_name}: {score}% ({len(matched)}/{len(CONTROL_DATASET)}) "
          f"[{risk_level(score)}] -- missing HIGH: {results[doc_name]['missing_high_risk']}")

print(f'\nMean score: {np.mean([r["score"] for r in results.values()]):.1f}%')

## 4. Compliance Score Summary Table

In [ ]:
rows = []
for doc_name, r in results.items():
    rows.append({
        'Document': doc_name,
        'Score (%)': r['score'],
        'Risk Level': r['risk_level'],
        'Controls Present': r['matched'],
        'Controls Missing': r['missing'],
        'Missing HIGH Risk': len(r['missing_high_risk']),
    })

df = pd.DataFrame(rows).sort_values('Score (%)', ascending=False)
df.reset_index(drop=True, inplace=True)
df

## 5. Gap Analysis: Systemic Missing Controls

In [ ]:
CONTROL_IDS = [c['id'] for c in CONTROL_DATASET]
RISK_MAP = {c['id']: c['risk'] for c in CONTROL_DATASET}
CTRL_NAME_MAP = {c['id']: c['control'] for c in CONTROL_DATASET}

missing_counts = {}
for cid in CONTROL_IDS:
    cnt = sum(1 for r in results.values() if cid in r['missing_ids'])
    missing_counts[cid] = cnt

gap_df = pd.DataFrame([
    {'Control ID': cid, 'Control Name': CTRL_NAME_MAP[cid],
     'Risk': RISK_MAP[cid], 'Policies Missing': missing_counts[cid],
     'Missing Rate (%)': round(missing_counts[cid]/len(results)*100, 0)}
    for cid in CONTROL_IDS
]).sort_values('Policies Missing', ascending=False)
gap_df.reset_index(drop=True, inplace=True)
print('Top 10 most frequently missing controls:')
gap_df.head(10)

## 6. Multi-LLM Routing: Cost Model

ProofLayer routes tasks to cost-appropriate models. This section demonstrates the cost model.

In [ ]:
# Cost per 1K tokens (approximate, mid-2025 pricing)
MODEL_COSTS = {
    'gpt-4o-mini':       {'input': 0.00015, 'output': 0.0006},
    'gpt-4o':            {'input': 0.005,   'output': 0.015},
    'claude-sonnet-4-6': {'input': 0.003,   'output': 0.015},
    'ollama/llama3':     {'input': 0.0,     'output': 0.0},
}

# Task routing table
ROUTING = {
    'extraction':        'gpt-4o-mini',
    'mapping':           'gpt-4o-mini',
    'gap_analysis':      'gpt-4o',
    'report_generation': 'gpt-4o',
    'long_doc (>15K)':   'claude-sonnet-4-6',
}

def estimate_cost(model, input_tokens, output_tokens):
    rates = MODEL_COSTS.get(model, {'input': 0.001, 'output': 0.003})
    return (input_tokens/1000)*rates['input'] + (output_tokens/1000)*rates['output']

# Corrected token counts (verified 2026-06-25):
# long-doc: claude-sonnet 6700 in, 1000 out = $0.035 (matches gpt-4o 4000in,1000out=$0.035)
# PL total: $0.086  Base total: $0.113  Savings: 23.9%

# Simulate a single-document audit (estimated token counts)
audit_steps = [
    ('extraction',        'gpt-4o-mini',     2000, 400),
    ('mapping',          'gpt-4o-mini',     1500, 300),
    ('gap_analysis',     'gpt-4o',          3000, 700),
    ('report_generation','gpt-4o',          2500, 800),
    ('long_doc (>15K)',  'claude-sonnet-4-6', 6700, 1000),  # full context; gpt-4o baseline uses 4000 in
]

proofLayer_cost = sum(estimate_cost(m, i, o) for _, m, i, o in audit_steps)
# Baseline: a single-model pipeline would not send the full long-document context
# to gpt-4o; it chunks to ~4000 input tokens for that step. Applying the documented
# baseline (see the note above) rather than reusing the 6700-token multi-LLM figure,
# which would otherwise overstate the saving as 32.0%.
LONG_DOC_BASELINE_INPUT = 4000
single_model_cost = sum(
    estimate_cost('gpt-4o', LONG_DOC_BASELINE_INPUT if t == 'long_doc (>15K)' else i, o)
    for t, m, i, o in audit_steps)

print(f'ProofLayer multi-LLM audit cost: ${proofLayer_cost:.4f}')
print(f'Single-model (gpt-4o) cost:      ${single_model_cost:.4f}')
print(f'Savings:                          {(1 - proofLayer_cost/single_model_cost)*100:.1f}%')
print()
print('Per-step breakdown:')
for task, model, inp, out in audit_steps:
    c = estimate_cost(model, inp, out)
    c_single = estimate_cost('gpt-4o', inp, out)
    print(f'  {task:<22} {model:<18} ${c:.5f}  (vs ${c_single:.5f} with gpt-4o)')

## 7. Figure Generation

Generates all paper figures as PDF files.

In [ ]:
# ── Fig 2: Overall compliance scores ────────────────────────────────────────
short_labels = [
    'Risk Mgmt\nPolicy', 'System &\nNetwork', 'Continuous\nMonitoring',
    'Cyber Council\nReport 2025', 'IT Security\nManual', 'Local Gov\nCyber 2021',
    'Health Dept\nIT Security', 'Judicial Info\nSecurity', 'MSDE AUP\n2024', 'Procurement\nManual',
]

scores = [results[k]['score'] for k in results]
colors = ['#e74c3c' if s < 40 else '#f39c12' if s < 60 else '#27ae60' for s in scores]

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(range(len(scores)), scores, color=colors, edgecolor='white', linewidth=0.8, width=0.65)
for bar, score in zip(bars, scores):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1.5,
            f'{score}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_xticks(range(len(short_labels)))
ax.set_xticklabels(short_labels, fontsize=8)
ax.set_ylabel('Compliance Coverage Score (%)')
ax.set_ylim(0, 105)
ax.axhline(80, color='green', linestyle='--', lw=1, alpha=0.6)
ax.axhline(55, color='orange', linestyle='--', lw=1, alpha=0.6)
plt.tight_layout()
plt.savefig('fig2_compliance_scores.pdf', bbox_inches='tight')
plt.show()
print('Figure 2 saved.')


In [ ]:
# ── Fig 3: Control coverage heatmap ─────────────────────────────────────────
matrix = np.array([
    [1 if cid in results[doc]['matched_ids'] else 0 for cid in CONTROL_IDS]
    for doc in results
])

fig, ax = plt.subplots(figsize=(13, 5.5))
im = ax.imshow(matrix, cmap=plt.cm.RdYlGn, aspect='auto', vmin=0, vmax=1)
ax.set_xticks(range(len(CONTROL_IDS)))
ax.set_xticklabels(CONTROL_IDS, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(len(short_labels)))
ax.set_yticklabels(short_labels, fontsize=8)
for i in range(len(results)):
    for j in range(len(CONTROL_IDS)):
        ax.text(j, i, 'P' if matrix[i,j] else 'X', ha='center', va='center',
                fontsize=6.5, color='black', fontweight='bold' if not matrix[i,j] else 'normal')
plt.colorbar(im, ax=ax, fraction=0.02, pad=0.01, label='Present (1) / Missing (0)')
plt.tight_layout()
plt.savefig('fig3_heatmap.pdf', bbox_inches='tight')
plt.show()
print('Figure 3 saved.')


In [ ]:
# ── Fig 5: Gap frequency across all policies ─────────────────────────────────
sorted_cids = sorted(CONTROL_IDS, key=lambda x: missing_counts[x], reverse=True)
miss_vals = [missing_counts[c] for c in sorted_cids]
bar_colors = ['#c0392b' if RISK_MAP[c]=='HIGH' else '#e67e22' for c in sorted_cids]

fig, ax = plt.subplots(figsize=(12, 4.5))
bars = ax.bar(range(len(sorted_cids)), miss_vals, color=bar_colors, edgecolor='white', width=0.7)
for bar, val in zip(bars, miss_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
            str(val), ha='center', va='bottom', fontsize=8, fontweight='bold')
ax.set_xticks(range(len(sorted_cids)))
ax.set_xticklabels(sorted_cids, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Number of Policies Missing Control (out of 10)')
ax.set_title('Figure 5: Frequency of Missing Controls Across All 10 Maryland Policy Documents')
red_p = plt.Rectangle((0,0),1,1, color='#c0392b')
orange_p = plt.Rectangle((0,0),1,1, color='#e67e22')
ax.legend([red_p, orange_p], ['HIGH Risk Control', 'MEDIUM Risk Control'], fontsize=9)
ax.set_ylim(0, 12)
plt.tight_layout()
plt.savefig('fig5_gap_frequency.pdf', bbox_inches='tight')
plt.show()
print('Figure 5 saved.')

## 8. Statistical Summary

In [ ]:
all_scores = [r['score'] for r in results.values()]
print('=== Compliance Score Statistics ===')
print(f'  Mean:    {np.mean(all_scores):.1f}%')
print(f'  Median:  {np.median(all_scores):.1f}%')
print(f'  Std Dev: {np.std(all_scores):.1f}%')
print(f'  Min:     {np.min(all_scores)}%  ({list(results.keys())[np.argmin(all_scores)]})')
print(f'  Max:     {np.max(all_scores)}%  ({list(results.keys())[np.argmax(all_scores)]})')

n_high = sum(1 for r in results.values() if r['risk_level']=='HIGH')
n_med  = sum(1 for r in results.values() if r['risk_level']=='MEDIUM')
n_low  = sum(1 for r in results.values() if r['risk_level']=='LOW')
print(f'\nRisk distribution: HIGH={n_high}, MEDIUM={n_med}, LOW={n_low}')

# Most commonly missing HIGH risk controls
high_risk_ids = [c['id'] for c in CONTROL_DATASET if c['risk']=='HIGH']
systemic_gaps = [(cid, missing_counts[cid]) for cid in high_risk_ids if missing_counts[cid] >= 5]
systemic_gaps.sort(key=lambda x: x[1], reverse=True)
print(f'\nSystemic HIGH-risk gaps (missing in >= 5 of 10 documents):')
for cid, cnt in systemic_gaps:
    print(f'  {cid}: {CTRL_NAME_MAP[cid]}  (missing in {cnt}/10 documents)')

## 9. Export Results as JSON

In [ ]:
import json
with open('proofLayer_maryland_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('Results exported to proofLayer_maryland_results.json')